In [ ]:
# CELL 1 - Authenticate and connect to BigQuery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = "YOUR-GCP-PROJECT-ID"   # <- confirm full ID first

client = bigquery.Client(project=PROJECT_ID)
print("Connected. Project:", client.project)

In [ ]:
# CELL 2 - Load and clean the physician resident progress notes cohort
query = """
SELECT ROW_ID, SUBJECT_ID, HADM_ID, CGID, CHARTDATE, TEXT
FROM `physionet-data.mimiciii_notes.noteevents`
WHERE TRIM(CATEGORY) = 'Physician'
  AND DESCRIPTION = 'Physician Resident Progress Note'
"""

print("Running cohort query, this may take a minute...")
notes = client.query(query).to_dataframe()
print(f"Loaded {len(notes)} notes.")
notes = notes.sort_values("ROW_ID").reset_index(drop=True)

# Clean immediately, in the same cell, so re-running cells out of order
# cannot leave the cohort in a half-cleaned state.
before = len(notes)
notes = notes.dropna(subset=["CGID", "HADM_ID", "TEXT"]).copy()
notes["CGID"] = notes["CGID"].astype(int)
notes["HADM_ID"] = notes["HADM_ID"].astype(int)
print(f"After dropping missing CGID/HADM_ID/TEXT: {len(notes)} "
      f"(removed {before - len(notes)})")
print(f"Distinct authors (CGID): {notes['CGID'].nunique()}")

In [ ]:
# CELL 3 - Preview the diagnoses table (schema check only, no full pull yet)
preview_query = """
SELECT *
FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
LIMIT 10
"""

preview = client.query(preview_query).to_dataframe()
print("Columns:", list(preview.columns))
print(preview.dtypes)
print()
preview

In [ ]:
# CELL 4 - Pull principal diagnoses (SEQ_NUM = 1) and merge with notes
dx_query = """
SELECT HADM_ID, SEQ_NUM, ICD9_CODE
FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
WHERE SEQ_NUM = 1
"""

print("Pulling principal diagnoses...")
dx = client.query(dx_query).to_dataframe()
dx = dx.dropna(subset=["HADM_ID", "ICD9_CODE"])
dx["HADM_ID"] = dx["HADM_ID"].astype(int)
dx["ICD9_CODE"] = dx["ICD9_CODE"].astype(str).str.strip().str.upper()

# one principal diagnosis per admission by definition, but guard anyway
dx = dx.drop_duplicates(subset=["HADM_ID"], keep="first")
print(f"Admissions with a principal diagnosis: {len(dx)}")

# merge onto the notes cohort (already cleaned and cast to int in the cleaning cell)
df = notes.merge(dx[["HADM_ID", "ICD9_CODE"]], on="HADM_ID", how="inner")
print(f"Notes matched to a principal diagnosis: {len(df)}")

In [ ]:
# CELL 5 - Word count, drop TEXT, rank top 10 diagnoses
import re

def word_count(text):
    # Letter-sequences only. Measures PROSE verbosity, not data density.
    # Deliberate: "Na 140 K 4.2" scores lower than the same content written
    # as full sentences. Documented here so it can be defended, not hidden.
    return len(re.findall(r"[a-zA-Z]+", str(text)))

def token_count_with_numbers(text):
    # Alternative that DOES count numbers and lab values.
    return len(re.findall(r"[a-zA-Z0-9]+", str(text)))

# Guard: only compute if TEXT still exists (protects against re-running)
if "TEXT" in df.columns:
    print("Counting words...")
    df["word_count"] = df["TEXT"].apply(word_count)
    df["token_count"] = df["TEXT"].apply(token_count_with_numbers)

    corr = df["word_count"].corr(df["token_count"])
    print(f"Correlation letters-only vs letters+numbers: {corr:.3f}")

    df = df.drop(columns=["TEXT"])
    print("TEXT column dropped. No raw note text remains in memory.")
else:
    print("TEXT already dropped (cell was re-run). word_count and token_count "
          "should already be in df.")
    if "word_count" not in df.columns:
        print("ERROR: word_count missing too. Restart runtime and re-run from Cell 1.")

TOP_N_DIAGNOSES = 10
counts = df["ICD9_CODE"].value_counts()
top_codes = list(counts.head(TOP_N_DIAGNOSES).index)

print(f"\nTop {TOP_N_DIAGNOSES} principal diagnoses by note count:")
for i, code in enumerate(top_codes, 1):
    sub = df[df["ICD9_CODE"] == code]
    print(f"{i:2d}. ICD9 {code:<8} "
          f"notes={counts[code]:>6}  authors={sub['CGID'].nunique():>4}")

In [ ]:
# ============================================================
# CANONICAL COHORT: defined ONCE, used by every downstream cell.
# Rule: authors with >= 5 notes across the pooled top-10 diagnoses.
# No further author filtering happens anywhere after this point.
# ============================================================
MIN_NOTES_PER_AUTHOR = 5

work = df[df["ICD9_CODE"].isin(top_codes)].copy()

n_before_author_filter = len(work)
authors_before = work["CGID"].nunique()

vc = work["CGID"].value_counts()
work = work[work["CGID"].isin(vc[vc >= MIN_NOTES_PER_AUTHOR].index)].copy()

print("=" * 60)
print("CANONICAL COHORT")
print("=" * 60)
print(f"Notes in top-10 diagnoses      : {n_before_author_filter}")
print(f"Authors before >=5 filter      : {authors_before}")
print(f"Notes after >=5 filter         : {len(work)}")
print(f"Authors after >=5 filter       : {work['CGID'].nunique()}")
print(f"Author attrition               : {authors_before - work['CGID'].nunique()} "
      f"({(1 - work['CGID'].nunique()/authors_before)*100:.1f}%)")
print(f"Note attrition                 : {n_before_author_filter - len(work)} "
      f"({(1 - len(work)/n_before_author_filter)*100:.1f}%)")

In [ ]:
# CELL 6 - Readable names + within/between physician analysis
import numpy as np

# --- readable diagnosis names (cosmetic; failure is not fatal) ---
icd_names = {}
try:
    name_q = "SELECT ICD9_CODE, SHORT_TITLE FROM `physionet-data.mimiciii_clinical.d_icd_diagnoses`"
    lookup = client.query(name_q).to_dataframe()
    lookup["ICD9_CODE"] = lookup["ICD9_CODE"].astype(str).str.strip().str.upper()
    icd_names = dict(zip(lookup["ICD9_CODE"], lookup["SHORT_TITLE"]))
    print(f"Loaded {len(icd_names)} diagnosis names")
except Exception as e:
    print(f"Name lookup unavailable ({e}); using codes only")

def pretty(code):
    nm = icd_names.get(code)
    return f"{code} {nm}" if nm else str(code)

# Author inclusion was fixed once in the canonical cohort cell.
# No further thresholds are applied here.

diagnosis_rows = []
physician_rows = []

for rank, code in enumerate(top_codes, 1):
    sub = work[work["ICD9_CODE"] == code].copy()   # canonical cohort, not df

    # STAGE 1: collapse each physician to ONE row (median + IQR)
    prof = (
        sub.groupby("CGID")["word_count"]
        .agg(n_notes="size", median_words="median",
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75))
        .reset_index()
    )
    prof["iqr"] = prof["q75"] - prof["q25"]
    prof["within_ratio"] = prof["q75"] / prof["q25"].clip(lower=1)

    # A physician with one note in this diagnosis has q25 == q75, so within_ratio
    # is 1.00 by construction and carries no information while dragging the median
    # toward 1.0. The RATIO comparison is restricted to physicians with >= 5 notes
    # in this diagnosis. The restriction and its cost are reported.
    MIN_NOTES_IN_DX = 5
    n_prof_all = len(prof)
    n_single   = int((prof["n_notes"] == 1).sum())
    prof_ratio = prof[prof["n_notes"] >= MIN_NOTES_IN_DX]

    # STAGE 2: between-physician spread, using median OF physician medians
    dx_reference = prof["median_words"].median()
    m = prof_ratio["median_words"]
    between_maxmin = m.max() / max(m.min(), 1)
    between_q75q25 = m.quantile(0.75) / max(m.quantile(0.25), 1)
    within_ratio_median = prof_ratio["within_ratio"].median()

    prof["icd_code"] = code
    prof["diagnosis"] = pretty(code)
    physician_rows.append(prof)

    diagnosis_rows.append({
        "rank": rank, "icd_code": code, "diagnosis": pretty(code),
        "n_notes": len(sub), "n_physicians": len(prof),
        "n_physicians_ratio": len(prof_ratio),
        "n_single_note": n_single,
        "reference_median_words": round(dx_reference, 1),
        "within_q75q25_restricted": round(within_ratio_median, 2),
        "between_q75q25_restricted": round(between_q75q25, 2),
        "between_maxmin_legacy": round(between_maxmin, 2),
    })

print("\nDone. Diagnoses that survived:", len(diagnosis_rows))

In [ ]:
# CELL 7 - Results table + headline verdict
import pandas as pd

dsum = pd.DataFrame(diagnosis_rows)
psum = pd.concat(physician_rows, ignore_index=True)

print("=" * 72)
print("RESULTS BY DIAGNOSIS")
print("=" * 72)
print(f"{'diagnosis':<42} {'notes':>6} {'docs':>5} {'within':>7} {'between':>8}")
print("-" * 72)
for _, r in dsum.iterrows():
    print(f"{r['diagnosis'][:42]:<42} {r['n_notes']:>6} "
          f"{r['n_physicians_ratio']:>5} {r['within_q75q25_restricted']:>6.2f}x "
          f"{r['between_q75q25_restricted']:>7.2f}x")

print()
print("HOW TO READ THIS")
print("  within  = the typical single physician's own spread on this diagnosis")
print("            (their 75th percentile note over their 25th). Near 1.0 = consistent.")
print("  between = 75th-percentile physician's median over 25th-percentile")
print("            physician's median, same diagnosis. Same percentile")
print("            definition as 'within', so the two are comparable.")

med_within  = dsum["within_q75q25_restricted"].median()
med_between = dsum["between_q75q25_restricted"].median()
print(f"Physicians per diagnosis, all        : {dsum['n_physicians'].sum()}")
print(f"Retained for ratio (>=5 notes in dx) : {dsum['n_physicians_ratio'].sum()}")
print(f"Single-note physicians excluded      : {dsum['n_single_note'].sum()}")
print(f"(legacy between max/min, NOT headline: {dsum['between_maxmin_legacy'].median():.2f}x)")

print("=" * 72)
print("HEADLINE")
print("=" * 72)
print(f"Median WITHIN-physician spread:  {med_within:.2f}x")
print(f"Median BETWEEN-physician spread: {med_between:.2f}x")
print()
gap = med_between - med_within
if abs(gap) < 0.05:
    print(f"Within ({med_within:.2f}x) and between ({med_between:.2f}x) are indistinguishable.")
    print("The ratio-based framing does NOT support an author-signature claim at")
    print("matched definitions. Use the ICC/variance-decomposition result instead.")
elif gap > 0:
    print(f"Between-physician variation exceeds within ({med_between:.2f}x vs {med_within:.2f}x).")
else:
    print(f"Within-physician variation is at least as large as between")
    print(f"({med_within:.2f}x vs {med_between:.2f}x). WEAKENS the signature claim.")

In [ ]:
# CELL 8 - Robustness: between-physician spread under three definitions
#
# Same quantity, three estimators, on the UNRESTRICTED physician set (every
# physician in the canonical cohort, regardless of how many notes they have in
# a given diagnosis). Cell 6/7 report the q75/q25 number on the RESTRICTED set
# (>= 5 notes in that diagnosis). Both are legitimate; they answer slightly
# different questions, so they are named differently and never mixed.
#
# max/min is an extreme-value statistic: two physicians decide it, and its
# sampling variance grows with cohort size by construction. p90/p10 and q75/q25
# are percentile statistics and are stable. That is the a priori reason for the
# switch, not an empirical claim - the correlations printed below are computed
# across only 10 diagnoses and are not distinguishable from zero.

rob = []
for code, g in psum.groupby("icd_code"):
    m = g["median_words"]
    rob.append({
        "diagnosis": g["diagnosis"].iloc[0],
        "n_docs": len(g),
        "between_maxmin":              m.max()          / max(m.min(), 1),
        "between_p90p10":              m.quantile(0.90) / max(m.quantile(0.10), 1),
        "between_q75q25_unrestricted": m.quantile(0.75) / max(m.quantile(0.25), 1),
    })

rob = pd.DataFrame(rob)
print(rob.to_string(index=False))

print()
print("BETWEEN-PHYSICIAN SPREAD, UNRESTRICTED SET")
print(f"  median max/min           : {rob['between_maxmin'].median():.2f}x   (fragile, not for headline)")
print(f"  median p90/p10           : {rob['between_p90p10'].median():.2f}x")
print(f"  median q75/q25           : {rob['between_q75q25_unrestricted'].median():.2f}x")

print()
print("Sensitivity of each estimator to cohort size (n=10 diagnoses, not significant):")
print(f"  n_docs vs max/min        : {rob['n_docs'].corr(rob['between_maxmin']):+.3f}")
print(f"  n_docs vs p90/p10        : {rob['n_docs'].corr(rob['between_p90p10']):+.3f}")
print(f"  n_docs vs q75/q25        : {rob['n_docs'].corr(rob['between_q75q25_unrestricted']):+.3f}")

In [ ]:
# CELL 8.5 - Variance decomposition and ICC (resolves the competing headline numbers)
import numpy as np
import pandas as pd

print(f"Notes: {len(work)}, diagnoses: {work['ICD9_CODE'].nunique()}, "
      f"authors: {work['CGID'].nunique()}")

# ---- Centering: remove the content effect by construction ----------------
# Subtract each diagnosis group's mean. Every group now has mean zero, so any
# surviving variance CANNOT be explained by which diagnosis the note is about.
work["group_mean"] = work.groupby("ICD9_CODE")["word_count"].transform("mean")
work["centered"] = work["word_count"] - work["group_mean"]

# ---- Simple decomposition -----------------------------------------------
author_offsets = work.groupby("CGID")["centered"].mean()
between_var = author_offsets.var()
work["author_offset"] = work["CGID"].map(author_offsets)
within_var = (work["centered"] - work["author_offset"]).var()
share = between_var / (between_var + within_var)

print("\n" + "=" * 60)
print("VARIANCE DECOMPOSITION (content held fixed by principal diagnosis)")
print("=" * 60)
print(f"Between-author variance: {between_var:,.0f}")
print(f"Within-author variance:  {within_var:,.0f}")
print(f"Author share:            {share*100:.1f}%")

# ---- Formal model: ICC ---------------------------------------------------
try:
    import statsmodels.formula.api as smf
    fit = smf.mixedlm("centered ~ 1", data=work, groups=work["CGID"]).fit(reml=True)
    var_a = float(fit.cov_re.iloc[0, 0])
    var_r = float(fit.scale)
    icc = var_a / (var_a + var_r)
    print(f"\nMixed model author variance: {var_a:,.1f}")
    print(f"Mixed model residual:        {var_r:,.1f}")
    print(f"ICC:                         {icc:.3f}  ({icc*100:.1f}%)")
    print(f"\nCross-check: simple {share*100:.1f}% vs model {icc*100:.1f}%")
except Exception as e:
    print(f"\nModel issue: {e}")
    icc = None

print("\n" + "=" * 60)
print("FALSIFICATION CHECK (5% floor; see limitations - this floor was too low)")
print("=" * 60)
if share < 0.05:
    print(f"Author share {share*100:.1f}% is below the 5% floor.")
    print("The author-signature claim is NOT supported. Report as stated.")
else:
    print(f"Author share {share*100:.1f}% is above the 5% floor.")
    print("The author-signature claim survives.")

In [ ]:
# CELL 9 - How clustered are notes within admissions?
notes_per_adm = work.groupby("HADM_ID").size()
adm_per_author = work.groupby("CGID")["HADM_ID"].nunique()
notes_per_author = work.groupby("CGID").size()

print(f"Notes per admission   median: {notes_per_adm.median():.1f}, "
      f"max: {notes_per_adm.max()}")
print(f"Admissions per author median: {adm_per_author.median():.1f}")
print(f"Notes per author      median: {notes_per_author.median():.1f}")
print(f"Notes-to-admissions ratio per author: "
      f"{(notes_per_author / adm_per_author).median():.2f}")

# Robustness: one note per admission per author, killing within-patient clustering
import statsmodels.formula.api as smf

def icc_one_per_admission(frame, how, seed=42):
    """One note per author-admission. how='first' (lowest ROW_ID) or 'random'."""
    if how == "first":
        sel = frame.sort_values("ROW_ID").groupby(["CGID", "HADM_ID"]).first().reset_index()
    else:
        sel = (frame.sample(frac=1.0, random_state=seed)
                    .groupby(["CGID", "HADM_ID"]).first().reset_index())
    sel = sel.copy()
    sel["group_mean"] = sel.groupby("ICD9_CODE")["word_count"].transform("mean")
    sel["centered"] = sel["word_count"] - sel["group_mean"]
    fit_x = smf.mixedlm("centered ~ 1", data=sel, groups=sel["CGID"]).fit(reml=True)
    va_x, vr_x = float(fit_x.cov_re.iloc[0, 0]), float(fit_x.scale)
    return va_x / (va_x + vr_x), len(sel), sel["CGID"].nunique()

icc2, n2, a2 = icc_one_per_admission(work, "first")
icc2r, n2r, a2r = icc_one_per_admission(work, "random", seed=42)
icc2r2, _, _ = icc_one_per_admission(work, "random", seed=7)
icc2r3, _, _ = icc_one_per_admission(work, "random", seed=2024)

print("\n" + "=" * 60)
print("COPY-FORWARD ROBUSTNESS: one note per admission per author")
print("=" * 60)
print(f"ICC, all notes                    : {icc:.3f}")
print(f"ICC, one-per-admission (first)    : {icc2:.3f}  (n={n2}, authors={a2})")
print(f"ICC, one-per-admission (random 42): {icc2r:.3f}  (n={n2r}, authors={a2r})")
print(f"ICC, one-per-admission (random 7) : {icc2r2:.3f}")
print(f"ICC, one-per-admission (random 2024): {icc2r3:.3f}")
print()
if abs(icc2 - icc2r) < 0.02:
    print("first and random agree -> selection rule is immaterial, report either.")
else:
    print("first and random DIFFER -> ROW_ID selection is not neutral.")
    print("Report the RANDOM version and disclose the discrepancy.")

# S2-E: full random-seed robustness sweep for the copy-forward draw
N_SEEDS = 20
vals = np.array([icc_one_per_admission(work, "random", seed=s)[0] for s in range(N_SEEDS)])
icc2_mean = vals.mean()
icc2_sd = vals.std(ddof=1)
icc2_lo, icc2_hi = np.percentile(vals, [5, 95])

print("\n" + "=" * 60)
print(f"RANDOM-DRAW ROBUSTNESS: {N_SEEDS} seeds")
print("=" * 60)
print(f"Mean ICC                : {icc2_mean:.3f}")
print(f"SD across seeds         : {icc2_sd:.3f}")
print(f"5th-95th percentile     : {icc2_lo:.3f} to {icc2_hi:.3f}")
print(f"Min / Max               : {vals.min():.3f} / {vals.max():.3f}")
print(f"Median ICC across seeds : {np.median(vals):.3f}")

# Does ROW_ID actually track chronology within an admission?
try:
    chk = (work.dropna(subset=["CHARTDATE"])
               .groupby("HADM_ID")
               .apply(lambda d: d["ROW_ID"].corr(d["CHARTDATE"].rank())
                      if len(d) > 2 else np.nan)
               .dropna())
    print(f"\nROW_ID vs chart-date rank correlation within admission:")
    print(f"  median={chk.median():.3f}, n_admissions={len(chk)}")
    print("  High (>0.8) means ROW_ID ~ chronological, so 'first' = admission-day note.")
except Exception as e:
    print(f"\nChronology check unavailable (CHARTDATE not in cohort): {e}")

In [ ]:
# CELL 10 - Headline figure, 2x2
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.formula.api as smf

# Per-diagnosis ICC: does the effect hold across all 10, or ride on one?
# Panel 3 needs enough physicians per diagnosis for a stable variance estimate.
# This is a DIAGNOSIS-level inclusion rule, not a second author filter.
# Any diagnosis it excludes is named below so Table 2 and Panel 3 reconcile.
MIN_PHYSICIANS_FOR_ICC = 5

per_dx, dx_excluded = [], []
for code in top_codes:
    d = work[work["ICD9_CODE"] == code].copy()   # no re-filtering
    if d["CGID"].nunique() < MIN_PHYSICIANS_FOR_ICC:
        dx_excluded.append((pretty(code), d["CGID"].nunique(), len(d)))
        continue
    d["centered"] = d["word_count"] - d["word_count"].mean()
    try:
        f = smf.mixedlm("centered ~ 1", data=d, groups=d["CGID"]).fit(reml=True)
        va, vr = float(f.cov_re.iloc[0,0]), float(f.scale)
        per_dx.append({"code": pretty(code)[:28], "icc": va/(va+vr),
                       "var_author": va, "var_resid": vr,
                       "n_notes": len(d), "n_docs": d["CGID"].nunique()})
    except Exception as e:
        print(f"per_dx failed for {code}: {e}")

per_dx = pd.DataFrame(per_dx).sort_values("icc")

print(f"Diagnoses in Table 2      : {len(top_codes)}")
print(f"Diagnoses in Panel 3 ICC  : {len(per_dx)}")
if dx_excluded:
    print(f"Excluded (<{MIN_PHYSICIANS_FOR_ICC} physicians):")
    for nm, nd, nn in dx_excluded:
        print(f"  {nm}: {nd} physicians, {nn} notes")
else:
    print("No diagnoses excluded. Table 2 and Panel 3 use the same 10.")

print("\nPER-DIAGNOSIS VARIANCE COMPONENTS")
print(per_dx[["code", "n_notes", "n_docs", "var_author", "var_resid", "icc"]]
      .to_string(index=False))
print(f"\nPooled ICC: {icc:.3f}")
print("If every per-diagnosis ICC exceeds pooled, check whether var_resid is")
print("systematically SMALLER within diagnosis than pooled. That is the mechanism.")
print(f"Pooled var_author={var_a:.1f}, var_resid={var_r:.1f}")
print(f"Median per-dx var_author={per_dx['var_author'].median():.1f}, "
      f"var_resid={per_dx['var_resid'].median():.1f}")

fig, ax = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Diagnosis-Anchored Author Variance in ICU Progress Notes",
             fontsize=14, fontweight="bold")

# 1. The headline split
a = ax[0,0]
a.barh(["Author", "Residual"], [icc, 1-icc],
       color=["#c62828", "#cfd8dc"], edgecolor="black")
a.set_xlim(0,1); a.set_xlabel("Share of content-adjusted variance")
a.set_title(f"ICC = {icc:.3f}  (admission-day note: {icc2:.3f})", fontweight="bold")
a.text(icc/2, 0, f"{icc*100:.1f}%", va="center", ha="center",
       color="white", fontweight="bold")

# 2. Caterpillar: every physician's baseline, ranked
a = ax[0,1]
off = author_offsets.sort_values()
a.plot(range(len(off)), off.values, color="#1565c0", lw=1.5)
a.fill_between(range(len(off)), 0, off.values, alpha=0.3, color="#1565c0")
a.axhline(0, color="red", ls="--", lw=1)
a.set_xlabel(f"Physicians, ranked ({len(off)} total)")
a.set_ylabel("Words vs diagnosis-typical")
a.set_title("Each physician's personal baseline")

# 3. Per-diagnosis ICC: robustness
a = ax[1,0]
a.barh(per_dx["code"], per_dx["icc"], color="#2e7d32", edgecolor="black")
a.axvline(icc, color="red", ls="--", label=f"pooled = {icc:.3f}")
a.set_xlabel("ICC within diagnosis")
a.set_title("Effect holds across diagnoses")
a.legend(fontsize=8); a.tick_params(labelsize=8)

# 4. Extremes vs middle
a = ax[1,1]
q = off.quantile([.1,.25,.5,.75,.9])
a.bar(["p10","p25","p50","p75","p90"], q.values, color=["#c62828","#ef9a9a","#cfd8dc","#90caf9","#1565c0"], edgecolor="black")
a.axhline(0, color="black", lw=1)
a.set_ylabel("Words vs diagnosis-typical")
a.set_title("Terse and verbose physicians differ by\n"
            f"{q[0.9]-q[0.1]:.0f} words at matched diagnosis")

plt.tight_layout(rect=[0,0,1,0.94])
plt.savefig("author_variance_icc.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CELL 13 - 95% CI for pooled ICC via cluster bootstrap (resample authors, not notes)
import time

B, SEED = 1000, 2026
rng = np.random.default_rng(SEED)
author_ids = work["CGID"].unique()
n_auth = len(author_ids)
groups_by_author = {cg: g for cg, g in work.groupby("CGID")}

icc_boot = []
t0 = time.time()
for b in range(B):
    sampled_ids = rng.choice(author_ids, size=n_auth, replace=True)
    frames = [groups_by_author[cg].assign(BOOT_ID=i) for i, cg in enumerate(sampled_ids)]
    boot_df = pd.concat(frames, ignore_index=True)
    boot_df["group_mean"] = boot_df.groupby("ICD9_CODE")["word_count"].transform("mean")
    boot_df["centered"] = boot_df["word_count"] - boot_df["group_mean"]
    try:
        f = smf.mixedlm("centered ~ 1", data=boot_df, groups=boot_df["BOOT_ID"]).fit(reml=True)
        va_b, vr_b = float(f.cov_re.iloc[0, 0]), float(f.scale)
        icc_boot.append(va_b / (va_b + vr_b))
    except Exception:
        continue

icc_boot = np.array(icc_boot)
lo, hi = np.percentile(icc_boot, [2.5, 97.5])

print("=" * 60)
print("95% CI FOR POOLED ICC (cluster bootstrap, resample authors)")
print("=" * 60)
print(f"Point estimate (locked)  : ICC = {icc:.4f}")
print(f"95% CI                   : {lo:.4f} to {hi:.4f}")
print(f"Bootstrap mean / SD      : {icc_boot.mean():.4f} / {icc_boot.std(ddof=1):.4f}")
print(f"Successful replicates    : {len(icc_boot)} / {B}")
print(f"Runtime                  : {time.time()-t0:.0f}s")
print(f"REPORT AS: ICC = {icc:.3f} (95% CI {lo:.3f} to {hi:.3f})")

In [ ]:
# ============================================================
# STUDY 2 — PAPER NUMBERS. Copy this whole output block.
# Every number that enters the manuscript is printed here, with the
# estimator and the sample it was computed on named explicitly.
# ============================================================
print("=" * 68)
print("STUDY 2 NUMBERS FOR MANUSCRIPT")
print("=" * 68)

print("\n-- COHORT --")
print(f"Notes in canonical cohort   : {len(work):,}")
print(f"Authors in canonical cohort : {work['CGID'].nunique()}")
print(f"Diagnoses                   : {work['ICD9_CODE'].nunique()}")
print(f"Notes per author, median    : {work.groupby('CGID').size().median():.0f}")

print("\n-- VARIANCE DECOMPOSITION (all notes, diagnosis-mean centered) --")
print(f"Simple decomposition share  : {share*100:.1f}%")
print(f"Mixed model var_author      : {var_a:,.1f}")
print(f"Mixed model var_resid       : {var_r:,.1f}")
print(f"ICC (HEADLINE)              : {icc:.4f}")

print("\n-- COPY-FORWARD ROBUSTNESS (one note per admission per author) --")
try:
    print(f"ICC across {len(vals)} random seeds  : mean {icc2_mean:.3f}, sd {icc2_sd:.3f}")
    print(f"  5th-95th percentile       : {icc2_lo:.3f} - {icc2_hi:.3f}")
    print(f"  min / max                 : {vals.min():.3f} / {vals.max():.3f}")
    print(f"  REPORT AS                 : ICC approximately {icc2_lo:.2f}-{icc2_hi:.2f}, "
          f"essentially unchanged from {icc:.3f}")
except NameError:
    print("  [twenty-seed block not run - apply S2-E]")
print(f"ICC, first note by ROW_ID   : {icc2:.4f}  (admission-day note)")
print("  ROW_ID tracks CHARTDATE (median rank r = 0.987), so 'first' is the")
print("  admission-day note. Higher ICC there is a DIRECTIONAL FINDING, not a")
print("  selection artefact: fresh notes carry more author signature than")
print("  copy-forward progress notes. State the interpretation as hypothesis.")

print("\n-- PHYSICIAN OFFSETS (words vs diagnosis-typical) --")
off = work.groupby("CGID")["centered"].mean().sort_values()
print(f"p10                         : {off.quantile(0.10):.0f} words")
print(f"p25                         : {off.quantile(0.25):.0f} words")
print(f"median                      : {off.median():.0f} words")
print(f"p75                         : {off.quantile(0.75):.0f} words")
print(f"p90                         : {off.quantile(0.90):.0f} words")
print(f"p90 - p10 gap (HEADLINE)    : {off.quantile(0.90)-off.quantile(0.10):.0f} words")
print(f"Full range (max - min)      : {off.max()-off.min():.0f} words  [fragile, not for headline]")

print("\n-- WITHIN vs BETWEEN PHYSICIAN, MATCHED DEFINITIONS --")
print("   Both q75/q25. Restricted to physicians with >= 5 notes in that diagnosis,")
print("   because a single-note physician has q25 == q75 and a within-ratio of")
print("   exactly 1.00 by construction.")
try:
    print(f"Physicians, all              : {dsum['n_physicians'].sum()}")
    print(f"Physicians retained for ratio: {dsum['n_physicians_ratio'].sum()}")
    print(f"Single-note excluded         : {dsum['n_single_note'].sum()}")
    print(f"Median WITHIN  q75/q25       : {dsum['within_q75q25_restricted'].median():.2f}x")
    print(f"Median BETWEEN q75/q25       : {dsum['between_q75q25_restricted'].median():.2f}x")
    print(f"Legacy between max/min       : {dsum['between_maxmin_legacy'].median():.2f}x  "
          f"[extreme-value, NOT for headline]")
except KeyError:
    print("  [restricted-ratio block not run - apply S2-D to Cell 6/7]")

print("\n-- BETWEEN-PHYSICIAN SPREAD, UNRESTRICTED SET --")
print("   Different sample from the block above. Do not compare the two directly.")
print(f"Median max/min               : {rob['between_maxmin'].median():.2f}x  [fragile]")
print(f"Median p90/p10               : {rob['between_p90p10'].median():.2f}x")
print(f"Median q75/q25               : {rob['between_q75q25_unrestricted'].median():.2f}x")

print("\n-- PER-DIAGNOSIS (Table 2) --")
cols = [c for c in ["code", "n_notes", "n_docs", "var_author", "var_resid", "icc"]
        if c in per_dx.columns]
print(per_dx[cols].to_string(index=False))
try:
    print(f"\nPooled     : var_author {var_a:,.1f}  var_resid {var_r:,.1f}  ICC {icc:.3f}")
    print(f"Median dx  : var_author {per_dx['var_author'].median():,.1f}  "
          f"var_resid {per_dx['var_resid'].median():,.1f}  "
          f"ICC {per_dx['icc'].median():.3f}")
    print("  Residual variance falls within diagnosis, as expected. Author variance")
    print("  RISES, because the pooled model averages each author's offsets across")
    print("  ten diagnoses toward zero. That is author-by-diagnosis interaction and")
    print("  it means 0.225 is the CONSERVATIVE estimate. Not formally tested - a")
    print("  random-slopes model would test it. Report as interpretation.")
except KeyError:
    print("  [variance components not stored - apply S2-FIX-2]")

print("=" * 68)